In [ ]:
# ar/statistics-101/hard/19-uncertainty-in-the-real-world
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


## ماذا يعني

كل فترة في هذا المقرر حتى الآن جاءت من صيغة تفترض شيئًا: الطبيعية، أو الاستقلال، أو شكلًا معلومًا. وعندما تفشل تلك الافتراضات، ما زال الحساب ينتج رقمًا، لكن الرقم لم يعد جديرًا بالثقة.

تتجنب **إعادة المعاينة** (bootstrap) الصيغة. لتقدير مقدار تأرجح إحصاءة ما، أعد معاينة بياناتك نفسها مع الإرجاع مرات كثيرة، وأعد حساب الإحصاءة على كل إعادة معاينة، وانظر إلى تشتت ما تحصل عليه:

$$
\text{resample} \to \hat{\theta}^* \quad\text{many times} \to \text{spread of } \hat{\theta}^* .
$$

فترة إعادة معاينة 95% هي الـ95% الوسطى من تلك القيم المعاد حسابها. ولا تستخدم أي افتراض توزيعي أبعد من "العينة تشبه المجتمع" — وهو الافتراض نفسه الذي يقدمه كل تقدير قائم على عينة.

## لماذا يضيف شيئًا

يمدد الفترات إلى إحصاءات لا صيغة بسيطة لها: الوسيط، والارتباط، ونسبة متوسطين، والفرق بين المئينين التسعينيين. فبدلًا من التنقيب عن صيغة في كتاب، تعيد المعاينة.

ويجعل معنى عدم اليقين ملموسًا أيضًا. تستطيع *رؤية* تباين التقدير بدلًا من الوثوق باشتقاق.

## مثال محسوب يدويًا، تصوريًا

وسيط $1,2,2,3,3,4,5,6,8,12,20,96$ هو 4.5. ما مدى عدم يقينه؟

لا يمكنك حساب ذلك من صيغة دون افتراض عن الشكل. أعد المعاينة:

1. اسحب 12 قيمة من البيانات، مع الإرجاع؛
2. احسب وسيط إعادة المعاينة تلك؛
3. كرر ذلك بضعة آلاف من المرات؛
4. خذ المئينين 2.5 و97.5 من الوسيطات.

الإرجاع مهم: دونه تكون كل إعادة معاينة هي البيانات الأصلية وكل وسيط متطابقًا.


In [ ]:
import random, statistics as st

hours = [1, 2, 2, 3, 3, 4, 5, 6, 8, 12, 20, 96]
medians = []

for _ in range(5000):
    resample = random.choices(hours, k=len(hours))   # with replacement
    medians.append(st.median(resample))

medians.sort()
lo, hi = medians[125], medians[-126]                # طرفا 2.5% و97.5%
print(round(st.median(hours), 1), round(lo, 1), round(hi, 1))


الفترة حول 4.5 واسعة — فمع 12 قيمة وحالة متطرفة واحدة، ليس الوسيط مثبتًا بقدر ما تدّعي صيغة تفترض الطبيعية.

<details class="challenge">
<summary>🧩 تحدٍّ، فكّر أولًا ثم أظهر</summary>
<div class="challenge__body">

أعد معاينة متوسط البيانات نفسها وقارن فترتها بفترة الوسيط. أيهما أوسع، ولماذا؟

<p class="challenge__answer">💡 <strong>الجواب:</strong> فترة المتوسط أوسع بكثير، لأن إعادة المعاينة تسحب القيمة 96 أحيانًا مرتين أو ثلاثًا فيقفز المتوسط وفقًا لذلك. أما الوسيط فلا يكاد يلاحظ. هذه هي حساسية القيم الشاذة نفسها التي رأيتها مع الانحراف المعياري، ظاهرةً الآن كعدم يقين.</p>

</div>
</details>

## ما لا تستطيعه إعادة المعاينة

- **لا يمكنها إصلاح التحيز.** إذا كانت العينة غير ممثلة، فإن إعادة معاينتها تعيد إنتاج الإجابة الخاطئة نفسها بفترة محكمة حولها.
- **تفترض أن العينة سحبة عادلة** من المجتمع الذي يهمك. ذلك الافتراض هو الذي يقوم بالعمل كله.
- **تتعثر مع العينات الصغيرة جدًا**، حيث لا يكون في إعادة المعاينة تنوع كافٍ للعمل عليه.
- **تفترض الاستقلال.** البيانات العنقودية تحتاج إعادة معاينة كتلية، والسلاسل الزمنية تحتاج حذرًا في الترتيب.

التمييز الذي يستحق الحفظ:

| | خطأ المعاينة | التحيز |
|---|---|---|
| ما هو | تأرجح عشوائي بين العينات | إزاحة منهجية عن الحقيقة |
| ما يصلحه | المزيد من البيانات | تصميم أفضل |
| ما يظهره | الفترات والأخطاء المعيارية | لا شيء عادةً — تبقى الفترات ضيقة |

فترة محكمة حول تقدير متحيز أخطر مخرج في الإحصاء، لأنها تشبه اليقين.

## الشيء نفسه في بايثون


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
hours = np.array([1, 2, 2, 3, 3, 4, 5, 6, 8, 12, 20, 96])

def bootstrap(stat, data, n=5000):
    idx = rng.integers(0, len(data), size=(n, len(data)))
    return np.array([stat(data[i]) for i in idx])

for name, fn in [('mean', np.mean), ('median', np.median)]:
    vals = bootstrap(fn, hours)
    print(name, np.round(np.percentile(vals, [2.5, 97.5]), 2))


تعمل الدالة نفسها لأي إحصاءة، وهذا هو السبب العملي لإبقائها في المتناول.

## كتابته

ملخص يستطيع المراجع أن يثق به يجيب عن هذه بالترتيب:

1. **ما كان السؤال**، مصوغًا قبل النظر إلى البيانات.
2. **ما قيس**، مع حجم العينة وكيفية سحبها.
3. **ما وُجد**، كأثر له فترة، لا كقيمة احتمالية وحدها.
4. **ما افتُرض**، بما في ذلك كل استبعاد وتعديل.
5. **ما قد يكون خاطئًا**، بتسمية التحيز الأكثر معقولية بدلًا من القول "نحتاج مزيدًا من البحث".
6. **ما يترتب على قرار**، مصوغًا بوحدات القرار.

## متى تستخدمه

- أي إحصاءة بلا صيغة نظيفة.
- أي مجموعة بيانات يفترضها التوزيع الطبيعي افتراضًا مشكوكًا فيه.
- كفحص متانة إلى جانب فترة مبنية على صيغة: إذا اتفقت الفترتان، فالاستنتاج أثبت؛ وإذا اختلفتا، فالافتراض كان حاملًا للحمل.

## القيود

- **فترات إعادة المعاينة ليست محصنة ضد العينات الصغيرة أو التحيز**، كما سبق.
- **قد تبخس عدم اليقين عندما تكون البيانات تابعة**، ما لم يحترم مخطط إعادة المعاينة التبعية.
- **إنها حسابية**، وهذا رخيص الآن لكنه يضيف أجزاء متحركة إلى التقرير.
- **لا تعوّض التفكير في التصميم.** لا مخطط إعادة معاينة يصلح عينة متحيزة.

## أخطاء شائعة

- **إعادة المعاينة دون إرجاع**، مما ينتج تغيرًا صفريًا وفترة تشبه نقطة.
- **استخدام إعادة المعاينة لتغطية عينة متحيزة.**
- **الإبلاغ عن التقدير النقطي وإسقاط الفترة**، مما يهدر المخرج كله.
- **اقتباس قيمة احتمالية بلا حجم أثر**، في أي كتابة.

## 🧩 تحديات

<details class="challenge">
<summary>🧩 تحدٍّ، فكّر أولًا ثم أظهر</summary>
<div class="challenge__body">

يبلغ استطلاع لزوار موقع عن "متوسط رضا 4.6 من 5، فترة ثقة 95% [4.55، 4.65]". لماذا قد تكون الفترة مضللة رغم ضيقها؟

<p class="challenge__answer">💡 <strong>الجواب:</strong> العينة هي الزوار — من ظلوا. ومن غادروا دون إجابة مفقودون، ورضاهم أدنى بكثير على نحو معقول. ذلك تحيز اختيار: تصف الفترة بدقة عدم يقين المتوسط بين المستجيبين، ولا تقول شيئًا عن مجتمع الزوار كله.</p>

</div>
</details>

## 🤔 أسئلة سقراطية

- لماذا تحاكي إعادة المعاينة مع الإرجاع سحب عينة جديدة؟
- أيهما أسوأ للإبلاغ: فترة واسعة أم فترة ضيقة حول تقدير متحيز؟
- ماذا تفترض فترة إعادة المعاينة مما لا تفترضه فترة الصيغة، والعكس؟
- إذا اختلفت فترة صيغة وفترة إعادة معاينة على البيانات نفسها، فأيهما تثق، وماذا تحقق؟

## ✅ فحص سريع

<div class="quiz" data-quiz="statistics-101-uncertainty-ar">
  <div class="quiz-q" data-answer="2">
    <p class="quiz-q__prompt">1. لماذا يجب أن تعيد إعادة المعاينة السحب مع الإرجاع؟</p>
    <div class="quiz-q__options">
      <button class="quiz-q__opt" data-idx="0">لجعل الحساب أسرع.</button>
      <button class="quiz-q__opt" data-idx="1">لأن الإحصاء يقتضي ذلك.</button>
      <button class="quiz-q__opt" data-idx="2">دون الإرجاع تكون كل إعادة معاينة هي البيانات الأصلية، فلا يوجد تباين لقياسه.</button>
      <button class="quiz-q__opt" data-idx="3">لإزالة القيم الشاذة.</button>
    </div>
    <p class="quiz-q__feedback" hidden></p>
  </div>
  <div class="quiz-q" data-answer="1">
    <p class="quiz-q__prompt">2. فترة ثقة ضيقة حول تقدير متحيز تخبرك ماذا؟</p>
    <div class="quiz-q__options">
      <button class="quiz-q__opt" data-idx="0">أن التقدير دقيق.</button>
      <button class="quiz-q__opt" data-idx="1">أن التقدير محكم، وأن التحيز خارج ما تقيسه الفترة.</button>
      <button class="quiz-q__opt" data-idx="2">أن العينة كبيرة بما يكفي لإزالة كل خطأ.</button>
      <button class="quiz-q__opt" data-idx="3">لا شيء على الإطلاق.</button>
    </div>
    <p class="quiz-q__feedback" hidden></p>
  </div>
  <p class="quiz__summary" data-quiz-summary hidden></p>
</div>


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
